In [1]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [3]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("../data/kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


### Let's divide the training and test set into two partitions

In [4]:
data.head()

,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0


In [5]:
# Your code
from sklearn.model_selection import train_test_split
X = data['text']
y = data['label']       #target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)


## Data Preprocessing

In [6]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [7]:
# Your code
import re

In [8]:
#Check what html patterns exist
html_patterns = ['<style', '<script', '<div', '<html']
for i, txt in enumerate(X_train):
    if any(pat in txt for pat in html_patterns):
        print(f"Sample {i}: {txt[:200]}...")

Sample 143: <HTML><BODY><div><div><TABLE cellSpacing=0 cellPadding=3 width="100%" bgColor=white border=0><TBODY><TR vAlign=top><TD width="100%"><FONT color=black size=2><TABLE cellSpacing=0 cellPadding=3 width="1...
Sample 326: <html><div style='background-color:'><DIV><DIV><DIV><DIV><DIV><DIV><P><STRONG>Dear Sir,/Madam ,</STRONG></P><P><STRONG>I am Princess&nbsp;Adama Williams, daughter of HRH King Solomon Abonmie Williams,...
Sample 342: <html><head><style>.hmmessage P{margin:0px;padding:0px}body.hmmessage...
Sample 383: <html><div style='background-color:'><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV><DIV class=RTE><DIV class=RTE>...
Sample 553: <html><head><style>P{margin:0px;padding:0px}body...
Sample 597: <html><head><style>P{margin:0px;padding:0px}body...


In [9]:
def clean_html(text):
    
    # Remove JavaScript y CSS (script y style)
    text = re.sub(r'<script.*?>.*?</script>', '', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'<style.*?>.*?</style>', '', text, flags=re.DOTALL | re.IGNORECASE)
    
    # Remove HTML comments
    text = re.sub(r'<!--.*?-->', '', text, flags=re.DOTALL)
    
    # Remove the remainign tags
    text = re.sub(r'<.*?>', '', text)
    
    return text

In [10]:
X_train_clean = X_train.apply(clean_html)
X_test_clean = X_test.apply(clean_html)

In [11]:
X_train_clean

29     ----------- REGARDS, MR NELSON SMITH.KINDLY RE...
535    I have not been able to reach oscar this am. W...
695    ; Huma Abedin B6I'm checking with Pat on the 5...
557    I can have it announced here on Monday - can't...
836        BANK OF AFRICAAGENCE SAN PEDRO14 BP 1210 S...
                             ...                        
106    7653 2612ADAMA IBRAHIM________________________...
270               What does that mean for our schedules?
860    Dear Friend,My Compliment to you,I guess this ...
435    Dear PRESIDENT=2FDIRECTOR=2C My name is Mr=2E ...
102    Let me know if today or tomorrow works for you...
Name: text, Length: 800, dtype: object

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [12]:
# Your code

# info from https://www.w3schools.com/python/python_regex.asp
def cleaner_text(text):

    # Remove special characters
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)     # ^ beggining from the string   # a-zA-Z0-9  all letters and numbers    #\s  space

    # Remove numbers
    text = re.sub(r'\d+', '', text)                 #\d  digit  # + one or more

    # Remove single characters
    text = re.sub(r'\b[a-zA-Z]\b', '', text)        # \b  limit of a word

    # Remove single characters from start
    text = re.sub(r'^[a-zA-Z]\s+', '', text)

    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Remove prefixed b
    text = re.sub(r"^b\s+", '', text)
    
    # Lowercase
    text = text.lower()
    
    return text

In [13]:
X_train_cleaner = X_train_clean.apply(cleaner_text)
X_test_cleaner = X_test_clean.apply(cleaner_text)

In [14]:
X_train_cleaner

29     regards mr nelson smith kindly reply me on my ...
535    have not been able to reach oscar this am we a...
695    huma abedin bi checking with pat on the will w...
557       can have it announced here on monday can today
836    bank of africaagence san pedro bp san pedro co...
                             ...                        
106    adama ibrahim tout savoir sur la curit de votr...
270                what does that mean for our schedules
860    dear friend my compliment to you guess this le...
435    dear president fdirector my name is mr micheal...
102    let me know if today or tomorrow works for you...
Name: text, Length: 800, dtype: object

## Now let's work on removing stopwords
Remove the stopwords.

In [15]:
# Your code
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Aitor\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Aitor\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [16]:
stop_words = set(stopwords.words('english'))

In [17]:
X_train_no_stopwords = []

for text in X_train_cleaner:
    
    tokens = word_tokenize(text)  # text into word list
    
    filtered_tokens = []
    
    for word in tokens:
        if word not in stop_words:
            filtered_tokens.append(word)
    
    cleaned_text = " ".join(filtered_tokens)
    
    X_train_no_stopwords.append(cleaned_text)

In [18]:
X_test_no_stopwords = []

for text in X_test_cleaner:
    
    tokens = word_tokenize(text)  # text into word list
    
    filtered_tokens = []
    
    for word in tokens:
        if word not in stop_words:
            filtered_tokens.append(word)
    
    cleaned_text = " ".join(filtered_tokens)
    
    X_test_no_stopwords.append(cleaned_text)

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [19]:
# Your code
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

In [20]:
X_train_lemmatized = []

for text in X_train_no_stopwords:
    
    tokens = word_tokenize(text)
    
    lemmatized_tokens = []
    
    for word in tokens:
        lemma = lemmatizer.lemmatize(word)
        lemmatized_tokens.append(lemma)
    
    lemmatized_text = " ".join(lemmatized_tokens)
    
    X_train_lemmatized.append(lemmatized_text)

In [21]:
X_test_lemmatized = []

for text in X_test_no_stopwords:
    
    tokens = word_tokenize(text)
    
    lemmatized_tokens = []
    
    for word in tokens:
        lemma = lemmatizer.lemmatize(word)
        lemmatized_tokens.append(lemma)
    
    lemmatized_text = " ".join(lemmatized_tokens)
    
    X_test_lemmatized.append(lemmatized_text)

## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [22]:
X_train_lemmatized

['regard mr nelson smith kindly reply private email address nelsonsmith yahoo com',
 'able reach oscar supposed send pdb receive',
 'huma abedin bi checking pat work jack jake rest also huma follow memo prep call',
 'announced monday today',
 'bank africaagence san pedro bp san pedro cote ivoire west africa dear sir mr dorise marie francoise accountant auditing accounting section bank africa cote ivoire due respect regard wish seek urgent assistance tr ansfer sum million dollar seven million five hundred th ousand dollar mentioned bank money belongs one customer robert rice died plane crash wife fund lying dormant bank claim eit family relation date contacting act bonafide next kin deceased th ere risk transaction loope hole taken care necessary information regard fund secu red notified still working bank colleague aware development thus treat proposal confidential secret security served bank faithfully good number year presen tly preparing retirement therefore see golden opportun ity 

In [23]:
# Your code
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

In [24]:
X_train_bag = vectorizer.fit_transform(X_train_lemmatized)
X_test_bag = vectorizer.transform(X_test_lemmatized)

In [25]:
train_bag_df = pd.DataFrame(X_train_bag.toarray(), columns=vectorizer.get_feature_names_out())
test_bag_df = pd.DataFrame(X_test_bag.toarray(), columns=vectorizer.get_feature_names_out())

train_bag_df['label'] = y_train.values
test_bag_df['label'] = y_test.values

In [26]:
#Ham = 0  Spam = 1
ham = train_bag_df[train_bag_df['label'] == 0]
top_ham = ham.drop(columns=['label']).sum().sort_values(ascending=False).head(10)

print("Top 10 words in Ham:")
print(top_ham)

spam = train_bag_df[train_bag_df['label'] == 1]
top_spam = spam.drop(columns=['label']).sum().sort_values(ascending=False).head(10)

print("\n Top 10 words in Spam:")
print(top_spam)

Top 10 words in Ham:
state        117
pm            97
would         94
mr            89
president     89
time          81
percent       80
obama         77
call          74
secretary     73
dtype: int64

 Top 10 words in Spam:
money          842
account        740
bank           645
fund           625
transaction    466
business       424
mr             420
country        419
million        366
company        365
dtype: int64


## Extra features

In [27]:
data_train = pd.DataFrame({'text': X_train,'label': y_train})
data_val = pd.DataFrame({'text': X_test,'label': y_test})

In [28]:
data_train['preprocessed_text'] = X_train_lemmatized
data_val['preprocessed_text'] = X_test_lemmatized

In [29]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

,text,label,preprocessed_text,money_mark,suspicious_words,text_len
29,"----------- REGARDS, MR NELSON SMITH.KINDLY RE...",1,regard mr nelson smith kindly reply private em...,0,0,79
535,I have not been able to reach oscar this am. W...,0,able reach oscar supposed send pdb receive,0,0,42
695,; Huma Abedin B6I'm checking with Pat on the 5...,0,huma abedin bi checking pat work jack jake res...,0,0,79
557,I can have it announced here on Monday - can't...,0,announced monday today,0,0,22
836,BANK OF AFRICAAGENCE SAN PEDRO14 BP 1210 S...,1,bank africaagence san pedro bp san pedro cote ...,1,1,1051


## How would work the Bag of Words with Count Vectorizer concept?

In [30]:
# Your code
vectorizer = CountVectorizer(min_df=5)
X_train_bow = vectorizer.fit_transform(data_train['preprocessed_text'])
X_test_bow = vectorizer.transform(data_val['preprocessed_text'])

In [31]:
X_train_bow_df = pd.DataFrame(
    X_train_bow.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=data_train.index
)

X_test_bow_df = pd.DataFrame(
    X_test_bow.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=data_val.index
)

In [33]:
X_train_final = pd.concat(
    [X_train_bow_df,
     data_train[['money_mark','suspicious_words','text_len']]],
    axis=1
)

X_test_final = pd.concat(
    [X_test_bow_df,
     data_val[['money_mark','suspicious_words','text_len']]],
    axis=1
)

## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [34]:
# Your code
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(data_train['preprocessed_text'])
X_test_tfidf = tfidf_vectorizer.transform(data_val['preprocessed_text'])
print("Train shape:", X_train_tfidf.shape)
print("Test shape:", X_test_tfidf.shape)

Train shape: (800, 20083)
Test shape: (200, 20083)


## And the Train a Classifier?

In [35]:
# Your code
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)

In [36]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.96

Confusion Matrix:
 [[125   0]
 [  8  67]]

Classification Report:
               precision    recall  f1-score   support

           0       0.94      1.00      0.97       125
           1       1.00      0.89      0.94        75

    accuracy                           0.96       200
   macro avg       0.97      0.95      0.96       200
weighted avg       0.96      0.96      0.96       200



### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [37]:
# Your code